# Run the finalized NewsQA benchmark

This notebook is a driver for the resumable CLI scripts. Each command writes to `reports/benchmarks/`, so interrupted cells can be rerun without repeating successful questions. Expensive generation and LLM judging are disabled in the configuration cell until explicitly enabled.

In [1]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env', override=False)
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['LANGSMITH_TRACING'] = 'false'
PYTHON = PROJECT_ROOT / '.venv/bin/python'
assert PYTHON.exists(), f'Virtual environment Python not found: {PYTHON}'
PROJECT_ROOT

PosixPath('/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG')

## Configuration

Use `QUESTION_VARIANT = 'original'` for the primary reported result and `QUESTION_VARIANT = 'resolved'` for the clarification treatment. Pilot and full inference must use different run directories because the selected question IDs are part of the run fingerprint.

In [2]:
QUESTION_VARIANT = 'resolved'       # 'original' or 'resolved'
RUN_MODE = 'full'                  # 'pilot' or 'full'
RETRIEVER = 'hybrid'                # 'dense', 'bm25', or 'hybrid'
RERANKER = 'cross-encoder'          # 'noop' or 'cross-encoder'
GENERATOR_MODEL = os.getenv('CHAT_MODEL') or ('deepseek-chat' if os.getenv('DEEPSEEK_API_KEY') else 'giahuytran4205/deepseek-v4-flash')
PILOT_QUESTIONS = 50
SEED = 42
MAX_ATTEMPTS = 3

JUDGE_PROVIDER = 'openai'           # OpenAI or the configured OpenAI-compatible gateway
JUDGE_MODEL = GENERATOR_MODEL       # Same model as generation; disclose self-judge bias
JUDGE_BATCH_SIZE = 5
JUDGE_MAX_WORKERS = 4
JUDGE_N_EVAL = None             # None = judge every successful saved answer
ALLOW_SAME_JUDGE = True

# Cost-bearing stages remain opt-in.
ENABLE_GENERATION = True
ENABLE_JUDGE = True
RETRY_EXHAUSTED = False

DATA_ROOT = PROJECT_ROOT / 'data/evaluation/newsqa_200_11064/final'
TESTSETS = {
    'original': DATA_ROOT / 'testset_reviewed_original.jsonl',
    'resolved': DATA_ROOT / 'testset_resolved.jsonl',
}
VARIANT_MANIFEST = PROJECT_ROOT / 'evaluation/manifests/newsqa_200_11064.variant.json'
RUN_NAME = f'{RUN_MODE}_{QUESTION_VARIANT}_{RETRIEVER}_{RERANKER}_{GENERATOR_MODEL}'.replace('/', '_')
RUN_DIR = PROJECT_ROOT / 'reports/benchmarks' / RUN_NAME

assert QUESTION_VARIANT in TESTSETS
assert RUN_MODE in {'pilot', 'full'}
print('Test set:', TESTSETS[QUESTION_VARIANT])
print('Run directory:', RUN_DIR)
print('DeepSeek key configured:', bool(os.getenv('DEEPSEEK_API_KEY')))
print('OpenAI key configured:', bool(os.getenv('OPENAI_API_KEY')))
print('Gemini key configured:', bool(os.getenv('GEMINI_API_KEY')))

Test set: /Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/data/evaluation/newsqa_200_11064/final/testset_resolved.jsonl
Run directory: /Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/reports/benchmarks/full_resolved_hybrid_cross-encoder_giahuytran4205_deepseek-v4-flash
DeepSeek key configured: False
OpenAI key configured: True
Gemini key configured: True


In [3]:
def run_command(arguments: list[str]) -> None:
    command = [str(PYTHON), *arguments]
    print('$', shlex.join(command))
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

def score_run(run_dir: Path) -> None:
    run_command(['scripts/score_benchmark_predictions.py', '--run-dir', str(run_dir)])

## 1. Corpus and index diagnostics

Set `SKIP_SELF_RETRIEVAL = True` if the local embedding model has not been downloaded. Structural and collection-count checks still run.

In [4]:
SKIP_SELF_RETRIEVAL = False
corpus_command = [
    'scripts/benchmark_corpus.py',
    '--variant-manifest', str(VARIANT_MANIFEST),
    '--output', str(PROJECT_ROOT / 'reports/benchmarks/newsqa_200_11064_corpus.json'),
]
if SKIP_SELF_RETRIEVAL:
    corpus_command.append('--skip-self-retrieval')
run_command(corpus_command)

$ '/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/.venv/bin/python' scripts/benchmark_corpus.py --variant-manifest '/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/evaluation/manifests/newsqa_200_11064.variant.json' --output '/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/reports/benchmarks/newsqa_200_11064_corpus.json'

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7146.27it/s]
Corpus report: /Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/reports/benchmarks/newsqa_200_11064_corpus.json
Index count matches manifest: True


## 2. Retrieval-only experiment matrix

This evaluates dense, BM25, and hybrid retrieval with both no-op and cross-encoder reranking. It makes no generator or judge API calls, but dense/hybrid and cross-encoder models may download from Hugging Face on first use. Set the flag to `True` before executing.

In [5]:
RUN_RETRIEVAL_MATRIX = False
RETRIEVAL_VARIANTS = ['original', 'resolved']
RETRIEVAL_CONFIGS = [
    ('dense', 'noop'),
    ('bm25', 'noop'),
    ('hybrid', 'noop'),
    ('dense', 'cross-encoder'),
    ('bm25', 'cross-encoder'),
    ('hybrid', 'cross-encoder'),
]

if RUN_RETRIEVAL_MATRIX:
    for question_variant in RETRIEVAL_VARIANTS:
        for retriever, reranker in RETRIEVAL_CONFIGS:
            run_dir = PROJECT_ROOT / 'reports/benchmarks' / f'{question_variant}_{retriever}_{reranker}_retrieval'
            command = [
                'scripts/collect_benchmark_predictions.py',
                '--retriever', retriever,
                '--reranker', reranker,
                '--retrieval-only',
                '--testset', str(TESTSETS[question_variant]),
                '--variant-manifest', str(VARIANT_MANIFEST),
                '--run-dir', str(run_dir),
                '--max-attempts', str(MAX_ATTEMPTS),
                '--progress',
            ]
            if RETRY_EXHAUSTED:
                command.append('--retry-failed')
            run_command(command)
            score_run(run_dir)
else:
    print('Retrieval matrix is disabled. Set RUN_RETRIEVAL_MATRIX = True to run it.')

Retrieval matrix is disabled. Set RUN_RETRIEVAL_MATRIX = True to run it.


## 3. Collect cited RAG answers

This is the cost-bearing generation stage. A valid provider key must be available in the notebook process. When `DEEPSEEK_API_KEY` is set, the project gateway selects DeepSeek generation. Rerun this cell unchanged to resume.

In [6]:
generation_command = [
    'scripts/collect_benchmark_predictions.py',
    '--retriever', RETRIEVER,
    '--reranker', RERANKER,
    '--generator-model', GENERATOR_MODEL,
    '--testset', str(TESTSETS[QUESTION_VARIANT]),
    '--variant-manifest', str(VARIANT_MANIFEST),
    '--run-dir', str(RUN_DIR),
    '--seed', str(SEED),
    '--max-attempts', str(MAX_ATTEMPTS),
    '--progress',
]
if RUN_MODE == 'pilot':
    generation_command.extend(['--n-eval', str(PILOT_QUESTIONS)])
if RETRY_EXHAUSTED:
    generation_command.append('--retry-failed')

if ENABLE_GENERATION and not (os.getenv('DEEPSEEK_API_KEY') or os.getenv('OPENAI_API_KEY')):
    raise RuntimeError('Configure DEEPSEEK_API_KEY or OPENAI_API_KEY for generation.')
if ENABLE_GENERATION:
    run_command(generation_command)
else:
    print('Generation is disabled. Set ENABLE_GENERATION = True after checking the run configuration and API key.')

$ '/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/.venv/bin/python' scripts/collect_benchmark_predictions.py --retriever hybrid --reranker cross-encoder --generator-model giahuytran4205/deepseek-v4-flash --testset '/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/data/evaluation/newsqa_200_11064/final/testset_resolved.jsonl' --variant-manifest '/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/evaluation/manifests/newsqa_200_11064.variant.json' --run-dir '/Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Mining---NewsQA-RAG/reports/benchmarks/full_resolved_hybrid_cross-encoder_giahuytran4205_deepseek-v4-flash' --seed 42 --max-attempts 3 --progress
Manifest preflight passed: /Users/thomas200905/Documents/Thomas/HCMUS/Third Year/Semester 9/Text Mining/project/Text-Min

KeyboardInterrupt: 

## 4. Deterministic scoring

This stage makes no API calls. Run it after collection and again after LLM judging to merge the cached judge results into `report.json`.

In [ ]:
if (RUN_DIR / 'predictions.jsonl').exists():
    score_run(RUN_DIR)
else:
    print('No predictions found yet:', RUN_DIR / 'predictions.jsonl')

## 5. LLM judge

Use a judge different from the generator for final reported scores. A pilot run judges at most 50 saved answers; a full run judges every successful saved answer. Rerun unchanged to resume.

In [ ]:
judge_command = [
    'scripts/judge_benchmark_predictions.py',
    '--run-dir', str(RUN_DIR),
    '--judge-provider', JUDGE_PROVIDER,
    '--judge-model', JUDGE_MODEL,
    '--batch-size', str(JUDGE_BATCH_SIZE),
    '--max-workers', str(JUDGE_MAX_WORKERS),
    '--max-attempts', str(MAX_ATTEMPTS),
    '--seed', str(SEED),
    '--progress',
]
if JUDGE_N_EVAL is not None:
    judge_command.extend(['--n-eval', str(JUDGE_N_EVAL)])
elif RUN_MODE == 'pilot':
    judge_command.extend(['--n-eval', str(PILOT_QUESTIONS)])
if RETRY_EXHAUSTED:
    judge_command.append('--retry-failed')
if ALLOW_SAME_JUDGE:
    judge_command.append('--allow-same-judge')

judge_key_names = {
    'openai': 'OPENAI_API_KEY',
    'deepseek': 'DEEPSEEK_API_KEY',
    'gemini': 'GEMINI_API_KEY',
}
judge_key_name = judge_key_names[JUDGE_PROVIDER]
if ENABLE_JUDGE and not os.getenv(judge_key_name):
    raise RuntimeError(f'{judge_key_name} is required for the selected judge provider.')
if ENABLE_JUDGE:
    run_command(judge_command)
    score_run(RUN_DIR)
else:
    print('LLM judging is disabled. Set ENABLE_JUDGE = True after checking the judge model and API key.')

## 6. Inspect the completed report

In [ ]:
report_path = RUN_DIR / 'report.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    summary_rows = {
        'coverage': report.get('coverage', {}),
        'retrieval_initial': report.get('retrieval_initial', {}),
        'retrieval_reranked': report.get('retrieval', {}),
        'reranker_delta': report.get('reranker_delta', {}),
        'qa': report.get('qa', {}),
        'citations': report.get('citations', {}),
        'ragas': report.get('ragas', {}),
        'latency_total': report.get('latency', {}).get('total', {}),
    }
    display(pd.DataFrame(summary_rows).T)
    failures = pd.DataFrame(report.get('failures', []))
    display(failures.head(20))
else:
    print('No report found yet:', report_path)